In [0]:
%sql
DROP TABLE IF EXISTS hive_metastore.f1_transformed.circuits;
DROP TABLE IF EXISTS hive_metastore.f1_transformed.drivers;
DROP TABLE IF EXISTS hive_metastore.f1_transformed.constructors;


In [0]:
%sql
DROP TABLE IF EXISTS hive_metastore.f1_transformed.drivers;


In [0]:

# COMMAND ----------
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, explode, current_timestamp
from delta.tables import DeltaTable
from pyspark.sql.functions import from_json, ArrayType

# COMMAND ----------
# 1. Outer Schema Enforcement
drivers_schema = StructType(fields=[
    StructField("MRData", StructType([
        StructField("DriverTable", StructType([
            StructField("Drivers", StringType(), True) 
        ]), True)
    ]), True)
])

raw_drivers_df = spark.read \
    .schema(drivers_schema) \
    .json("dbfs:/mnt/f1-raw/drivers/*")

# COMMAND ----------
# 2. Inner Array Schema Definition & Transformation
driver_element_schema = StructType([
    StructField("driverId", StringType(), False),
    StructField("permanentNumber", StringType(), True),
    StructField("code", StringType(), True),
    StructField("givenName", StringType(), True),
    StructField("familyName", StringType(), True),
    StructField("dateOfBirth", StringType(), True),
    StructField("nationality", StringType(), True)
])

parsed_df = raw_drivers_df.withColumn(
    "driver_array", 
    from_json(col("MRData.DriverTable.Drivers"), ArrayType(driver_element_schema))
)
exploded_df = parsed_df.select(explode(col("driver_array")).alias("driver"))

silver_drivers_df = exploded_df.select(
    col("driver.driverId").alias("driver_id"),
    col("driver.permanentNumber").cast("int").alias("driver_number"),
    col("driver.code").alias("driver_code"),
    col("driver.givenName").alias("first_name"),
    col("driver.familyName").alias("last_name"),
    col("driver.dateOfBirth").cast("date").alias("dob"),
    col("driver.nationality").alias("nationality"),
    current_timestamp().alias("ingestion_date")
).dropDuplicates(["driver_id"])

display(silver_drivers_df)

# COMMAND ----------
# 3. Hive Metastore Compliant Table Write & Upsert
target_path = "dbfs:/mnt/f1-transformed/drivers"

# 1. Force clear the internal metastore cache database
spark.sql("CREATE DATABASE IF NOT EXISTS hive_metastore.f1_transformed")

# 2. CLEAR CORRUPTED CACHE: If the table exists, we drop it to fix the null fields issue
# Run this once to clean out the old schema definition completely
if spark.catalog.tableExists("hive_metastore.f1_transformed.drivers"):
    spark.sql("DROP TABLE hive_metastore.f1_transformed.drivers")
    # Clean up the DBFS metadata pointer layer
    dbutils.fs.rm(target_path, recurse=True)

# 3. Re-initialize the clean schema with explicit path routing
if not spark.catalog.tableExists("hive_metastore.f1_transformed.drivers"):
    silver_drivers_df.write \
        .format("delta") \
        .option("path", target_path) \
        .mode("overwrite") \
        .saveAsTable("hive_metastore.f1_transformed.drivers")
    print("Drivers table successfully initialized and registered inside your Azure container.")
else:
    # Production Merge Routine (Will execute cleanly on your second run)
    tgt_table = DeltaTable.forName(spark, "hive_metastore.f1_transformed.drivers")
    tgt_table.alias("tgt") \
        .merge(
            source = silver_drivers_df.alias("src"), 
            condition = "tgt.driver_id = src.driver_id"
        ) \
        .whenMatchedUpdate(set = {
            "driver_number": "src.driver_number", 
            "driver_code": "src.driver_code", 
            "first_name": "src.first_name", 
            "last_name": "src.last_name", 
            "dob": "src.dob", 
            "nationality": "src.nationality", 
            "ingestion_date": "src.ingestion_date"
        }) \
        .whenNotMatchedInsert(values = {
            "driver_id": "src.driver_id", 
            "driver_number": "src.driver_number", 
            "driver_code": "src.driver_code", 
            "first_name": "src.first_name", 
            "last_name": "src.last_name", 
            "dob": "src.dob", 
            "nationality": "src.nationality", 
            "ingestion_date": "src.ingestion_date"
        }) \
        .execute()
    print("Drivers incremental update completed successfully via Hive Metastore Delta Merge.")
